# 06. Modelo productivo (pyfunc) y MLflow remoto con DagsHub

**Proyecto**: Pronóstico de demanda multi-series (Curso II — Especialización ML Engineering)

Este notebook cubre la **Fase 4** en dos partes:

1. **Parte local**: cargar el modelo de producción `models:/demand_forecast@Production`
   (cualquiera que haya ganado según la regla MASE + WAPE con filtro de sesgo),
   verificar su envoltura **pyfunc**, predecir el periodo **ene-mar 2018**
   (`data/raw/test.csv`) y guardar la submission en formato Kaggle.
2. **Parte remota (DagsHub)**: conectar MLflow a DagsHub para tener el **link público
   de evidencia** de experimentos (requisito del entregable). Requiere crear cuenta,
   repo y token en DagsHub; si no hay credenciales esta parte se omite.

> **Alcance**: el proyecto trabaja sobre el **subconjunto de 150 series** (10 tiendas ×
> 15 artículos, decisión de la sesión 2 para cumplir el límite <100 MB del curso).
> `test.csv` tiene 45,000 filas (500 series), pero aquí se predicen **solo las 150 series
> del subconjunto** (13,500 filas). La submission es **demostrativa del subconjunto**,
> no apta para el leaderboard de la competición.

> Reutiliza los módulos de `src/` (`build_features`, `configs`, `train_model`) para
> que la lógica siga viviendo en un solo lugar.

> **Nota para máquina nueva**: los experimentos de la fase 3 se registraron en la base
> local `mlruns/mlflow.db` de la otra PC. Si aquí no existe el alias `Production`, este
> notebook lo re-entrena con el mismo config de LightGBM (v9) y lo registra.

In [1]:
%matplotlib inline
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from mlflow.tracking import MlflowClient

# Localiza la raíz del proyecto estés donde estés (desde notebooks/ o desde la raíz)
def _find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "raw" / "train.csv").exists():
            return p
    raise FileNotFoundError("No se encontró la raíz del proyecto (data/raw/train.csv)")

ROOT = _find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

from src.features.build_features import build_features, FEATURE_COLUMNS  # noqa: E402
from src.models.metrics import naive_scale_by_series  # noqa: E402
from src.models.train_model import (  # noqa: E402
    ModelConfig,
    _log_sklearn_model,
    promote_best_model,
    train_and_log_model,
)

sns.set_theme()
pd.set_option("display.float_format", "{:.2f}".format)

DATA_RAW = ROOT / "data" / "raw"
DATA_OUT = ROOT / "data" / "processed"
FIGURES = ROOT / "reports" / "figures"
SUBMISSIONS = ROOT / "reports" / "submissions"
FIGURES.mkdir(parents=True, exist_ok=True)
SUBMISSIONS.mkdir(parents=True, exist_ok=True)

TRACKING_URI = f"sqlite:///{(ROOT / 'mlruns' / 'mlflow.db').as_posix()}"
REGISTERED_MODEL = "demand_forecast"
EXPERIMENT_NAME = "demand_forecast_fase3"
TARGET = "sales"
TRAIN_CUTOFF = "2017-09-30"

DEFAULT_ITEMS = [1, 2, 5, 6, 7, 8, 13, 14, 15, 16, 23, 24, 25, 28, 49]
DEFAULT_STORES = list(range(1, 11))

mlflow.set_tracking_uri(TRACKING_URI)
# El kernel de Jupyter corre con cwd=notebooks/; fijamos el root de artefactos
# para que los runs que se creen aquí caigan en mlruns/ de la raíz, no en notebooks/mlruns
import os  # noqa: E402

os.environ["MLFLOW_DEFAULT_ARTIFACT_ROOT"] = str(ROOT / "mlruns")
print("OK")

C:\Users\USER\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OK


## 1. Modelo de producción local

El Model Registry guarda las versiones registradas de `demand_forecast`; el alias
`Production` apunta a la mejor según la regla de decisión. Se comprueba si existe
localmente y, si no, se re-entrena LightGBM para poder continuar.

In [2]:
client = MlflowClient()

def production_version() -> int | None:
    try:
        return int(client.get_model_version_by_alias(REGISTERED_MODEL, "Production").version)
    except Exception:
        return None

prod_version = production_version()
print(f"Production local disponible: {prod_version is not None}")
if prod_version is not None:
    mv = client.get_model_version(REGISTERED_MODEL, prod_version)
    print(f"  v{prod_version} | status={mv.status} | aliases={mv.aliases}")

Production local disponible: True
  v1 | status=READY | aliases=['Production']


### 1.1 Re-entrenar Production si no existe (máquina nueva)

En una clonación fresca la base `mlruns/mlflow.db` está vacía (los runs de la fase 3
se registraron en la otra PC). Se reproduce el flujo de `scripts/train.py` solo con
LightGBM, el ganador de la fase 3 (MASE=0.581, WAPE=10.41%, rel_bias=-0.25%), y se
promueve a `Production` con la misma regla.

In [3]:
if production_version() is None:
    print("Sin Production local: re-entrenando LightGBM (config de fase 3)...")
    train = pd.read_csv(DATA_OUT / "train_features.csv")
    holdout = pd.read_csv(DATA_OUT / "holdout_features.csv")
    train = train.dropna().reset_index(drop=True)
    holdout = holdout.dropna().reset_index(drop=True)

    features = [c for c in FEATURE_COLUMNS if c in train.columns]
    X_train, y_train = train[features], train[TARGET]
    X_test, y_test = holdout[features], holdout[TARGET]
    df_test = holdout[["date", "store", "item", "sales"]].reset_index(drop=True)
    series_ids_test = pd.Series(list(zip(df_test["store"], df_test["item"])))
    train_raw = pd.read_csv(DATA_OUT / "train.csv")
    naive_scale = naive_scale_by_series(train_raw)

    mlflow.set_experiment(EXPERIMENT_NAME)
    from lightgbm import LGBMRegressor  # noqa: E402

    lgb_params = {
        "n_estimators": 300, "learning_rate": 0.05, "num_leaves": 63,
        "random_state": 42, "n_jobs": -1, "verbose": -1,
    }
    cfg = ModelConfig(
        name="lightgbm", family="gbm",
        estimator_factory=lambda: LGBMRegressor(**lgb_params),
        params=dict(lgb_params),
    )
    train_and_log_model(
        cfg,
        X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
        df_test=df_test, series_ids_test=series_ids_test, naive_scale=naive_scale,
        experiment_name=EXPERIMENT_NAME, registered_model_name=REGISTERED_MODEL,
        feature_names=list(X_train.columns),
    )
    promote_best_model(EXPERIMENT_NAME, REGISTERED_MODEL, alias="Production")
    print("Production re-entrenado: v" + str(production_version()))
else:
    print("Production ya existe; no se re-entrena.")

Production ya existe; no se re-entrena.


## 2. Consumir el modelo productivo como pyfunc

Cada versión registrada guarda el artefacto `model/` con un `MLmodel` que declara las
flavors. Como se registró con `mlflow.sklearn.log_model`, incluye la flavor `pyfunc`
(envoltura estándar) además de `sklearn`. Cargar por `models:/<nombre>@Production`
devuelve la envoltura `pyfunc` sin necesidad de saber qué modelo ganó.

In [4]:
model = mlflow.pyfunc.load_model(f"models:/{REGISTERED_MODEL}@Production")
print("Tipo del objeto cargado:", type(model).__name__)
print("PyfuncModel cargado correctamente.")

Tipo del objeto cargado: PyFuncModel
PyfuncModel cargado correctamente.


In [5]:
from mlflow.store.artifact.models_artifact_repo import ModelsArtifactRepository

# Descarga el artefacto del modelo registrado para inspeccionar el MLmodel
art_dir = Path(ModelsArtifactRepository(
    f"models:/{REGISTERED_MODEL}@Production").download_artifacts(""))
print("Artefacto local:", art_dir)
print()
print(art_dir.joinpath("MLmodel").read_text(encoding="utf-8"))

Artefacto local: D:\Jaime Ramos 2\00 Entorno Visual code\ML2_Series_de_tiempo\notebooks\mlruns\2\models\m-7a8fc7bd995646809462974ad56e4e5d\artifacts

artifact_path: file:D:/Jaime Ramos 2/00 Entorno Visual code/ML2_Series_de_tiempo/notebooks/mlruns/2/models/m-7a8fc7bd995646809462974ad56e4e5d/artifacts
flavors:
  python_function:
    env:
      conda: conda.yaml
      virtualenv: python_env.yaml
    loader_module: mlflow.sklearn
    model_path: model.skops
    predict_fn: predict
    python_version: 3.14.5
  sklearn:
    code: null
    pickled_model: model.skops
    serialization_format: skops
    sklearn_version: 1.9.0
    skops_trusted_types:
    - collections.OrderedDict
    - lightgbm.basic.Booster
    - lightgbm.sklearn.LGBMRegressor
mlflow_version: 3.15.1
model_id: m-7a8fc7bd995646809462974ad56e4e5d
model_size_bytes: 1763711
model_uuid: m-7a8fc7bd995646809462974ad56e4e5d
prompts: null
run_id: 0ca4a1eff4934430ae13392505fe382e
utc_time_created: '2026-08-12 17:09:59.490442'



## 3. Features del periodo a pronosticar (ene-mar 2018)

`data/raw/test.csv` cubre 2018-01-01 a 2018-03-31 (90 días por serie) sin la columna
`sales`. Como las features del modelo son **día-relativas** (`lag_1` = ventas de ayer,
`rolling_*` = historial reciente), predecir los 90 días de una sola vez dejaría
`lag_1`/`rolling_*` con NaN a partir del segundo día (dependerían del target
desconocido).

Solución usada aquí: **pronóstico recursivo**. Se arma el marco *historial + test* y se
predice **día a día**: cada día se rellenan las ventas predichas en la serie para que el
día siguiente tenga lags/rolling válidos (compuestos sobre historial real + predicciones
previas). Es la estrategia estándar para modelos autoregresivos de horizonte largo.

> Alternativa más robusta (fuera de alcance): un modelo *directo* multi-horizonte con
> una salida por día, o features ancladas al inicio de la ventana (sin recursión).

In [6]:
def build_forecast_frame() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Historial (150 series) + test con `sales` en NaN para el periodo futuro."""
    train_raw = pd.read_csv(DATA_RAW / "train.csv", parse_dates=["date"])
    test = pd.read_csv(DATA_RAW / "test.csv", parse_dates=["date"])

    hist = train_raw[
        train_raw["store"].isin(DEFAULT_STORES) & train_raw["item"].isin(DEFAULT_ITEMS)
    ][["date", "store", "item", "sales"]].copy()
    tst = test[
        test["store"].isin(DEFAULT_STORES) & test["item"].isin(DEFAULT_ITEMS)
    ][["id", "date", "store", "item"]].copy()

    frame = pd.concat(
        [hist, tst.assign(sales=np.nan)[["date", "store", "item", "sales"]]],
        ignore_index=True,
    ).sort_values(["store", "item", "date"]).reset_index(drop=True)
    return frame, tst

frame, tst = build_forecast_frame()
print(f"Historial: {len(frame[frame['sales'].notna()]):,} filas | "
      f"A pronosticar: {frame['sales'].isna().sum():,} filas "
      f"({tst['date'].nunique()} días x {tst['store'].nunique()} tiendas x "
      f"{tst['item'].nunique()} items)")

Historial: 273,900 filas | A pronosticar: 13,500 filas (90 días x 10 tiendas x 15 items)


## 4. Predicción recursiva y submission

Se itera por día sobre `forecast_dates`: se reconstruyen las features sobre el marco
actual (las ventas ya predichas alimentan los lags del día siguiente), se predice con la
envoltura `pyfunc` y se rellenan las ventas. Al final se cruzan las predicciones con el
`id` de `test.csv` y se guarda la submission con el formato del `sample_submission.csv`
(`id, sales`).

> La submission cubre **solo las 150 series del subconjunto** (13,500 de las 45,000
> filas del `sample_submission.csv`), coherente con el alcance del proyecto.

In [7]:
probe = build_features(frame, target=TARGET)
FEAT = [c for c in FEATURE_COLUMNS if c in probe.columns]
forecast_dates = sorted(frame.loc[frame["sales"].isna(), "date"].unique())
print(f"Features del modelo: {len(FEAT)} ({', '.join(FEAT)})")

for day, d in enumerate(forecast_dates, 1):
    feat = build_features(frame, target=TARGET)
    idx = feat.index[feat["date"] == d]
    frame.loc[idx, "sales"] = np.maximum(model.predict(feat.loc[idx, FEAT]), 0.0)
    if day % 30 == 0:
        print(f"  día {day:2d}/90 ({d.date()}) predicho")

pred_df = frame[frame["date"].isin(forecast_dates)].merge(
    tst, on=["date", "store", "item"], how="inner"
)
assert len(pred_df) == len(tst), "Deben cubrirse todas las filas de test"

submission = pred_df[["id", "sales"]].sort_values("id").reset_index(drop=True)
submission["id"] = submission["id"].astype(int)
sub_path = SUBMISSIONS / "submission_production.csv"
submission.to_csv(sub_path, index=False)
print(f"Guardada: {sub_path}")
print(f"Filas: {len(submission):,} | Media diaria: {submission['sales'].mean():.1f} unidades")
submission.head(10)

Features del modelo: 15 (year, month, day, dayofweek, weekofyear, dayofyear, store, item, lag_1, lag_7, lag_30, rolling_mean_7, rolling_mean_30, rolling_std_7, rolling_std_30)


  día 30/90 (2018-01-30) predicho


  día 60/90 (2018-03-01) predicho


  día 90/90 (2018-03-31) predicho
Guardada: D:\Jaime Ramos 2\00 Entorno Visual code\ML2_Series_de_tiempo\reports\submissions\submission_production.csv
Filas: 13,500 | Media diaria: 52.5 unidades


,id,sales
0,0,13.05
1,1,15.25
2,2,14.73
3,3,16.01
4,4,17.34
5,5,18.51
6,6,19.19
7,7,12.79
8,8,14.67
9,9,14.87


In [8]:
fig, ax = plt.subplots(figsize=(11, 4))
agg = pred_df.groupby("date")["sales"].sum()
ax.plot(agg.index, agg.values, label="Pronóstico (Production)", marker="o", ms=3, lw=1.2)
ax.set_title("Pronóstico agregado diario — ene-mar 2018 (150 series)")
ax.set_xlabel("Fecha")
ax.set_ylabel("Unidades")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "forecast_ene_mar_2018.png", dpi=150)
plt.show()

C:\Users\USER\AppData\Local\Temp\ipykernel_16136\777654988.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. MLflow remoto en DagsHub (evidencia pública)

El entregable pide un **link público de experimentos en MLflow** con artefactos y un
modelo productivo. Plan gratuito con DagsHub:

1. Crear cuenta en https://dagshub.com (este proyecto usa el usuario `jaimeramos124`).
2. Crear un repositorio (en este caso `ML2_Series_de_tiempo`).
3. Token: *Settings → User settings → Tokens* → crear `DAGSHUB_TOKEN`.
4. Guardar en `.env` (gitignoreado):
   ```env
   DAGSHUB_OWNER=<usuario de DagsHub>
   DAGSHUB_REPO=<nombre del repo en DagsHub>
   DAGSHUB_TOKEN=<token>
   ```
5. `dagshub.init(owner, repo, mlflow=True)` apunta MLflow al remoto; los runs que se
   creen después quedan en DagsHub (ver URL del experimento en la salida).

> La celda siguiente **no se ejecuta** si faltan las credenciales. Además del run de
> ejemplo, registra el **modelo de producción** (`demand_forecast`) en el Model Registry
> remoto con el alias `Production`, subiendo también la submission como artefacto.

In [9]:
from dotenv import dotenv_values

env = dotenv_values(ROOT / ".env") if (ROOT / ".env").exists() else {}
DAGSHUB_OWNER = env.get("DAGSHUB_OWNER", "jaimeramos124")
DAGSHUB_REPO = env.get("DAGSHUB_REPO", "")
DAGSHUB_TOKEN = env.get("DAGSHUB_TOKEN", "")

if DAGSHUB_TOKEN and DAGSHUB_REPO:
    import os
    import time

    import dagshub

    # 1) Capturar modelo y métricas del Production LOCAL antes de cambiar el tracking
    local_sk = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL}@Production")
    local_mv = client.get_model_version_by_alias(REGISTERED_MODEL, "Production")
    local_run = client.get_run(local_mv.run_id)
    remote_metrics = {
        "mase": local_run.data.metrics.get("mase", 0.581),
        "wape": local_run.data.metrics.get("wape", 10.41),
        "rel_bias_pct": local_run.data.metrics.get("rel_bias_pct", -0.25),
    }

    # 2) Cambiar el tracking a DagsHub
    os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_OWNER
    os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
    dagshub.init(DAGSHUB_REPO, DAGSHUB_OWNER, mlflow=True)

    mlflow.set_experiment(EXPERIMENT_NAME)
    with mlflow.start_run(run_name="fase4_remoto_production"):
        mlflow.log_params({
            "entorno": "DagsHub",
            "modelo": REGISTERED_MODEL,
            "version_local": str(local_mv.version),
            "submission": sub_path.name,
        })
        mlflow.log_metrics(remote_metrics)
        mlflow.log_artifact(str(sub_path))
        _log_sklearn_model(local_sk)
        remote_run_id = mlflow.active_run().info.run_id

    registered = mlflow.register_model(
        model_uri=f"runs:/{remote_run_id}/model", name=REGISTERED_MODEL
    )
    remote_client = MlflowClient()
    for _ in range(40):
        if remote_client.get_model_version(REGISTERED_MODEL, registered.version).status == "READY":
            break
        time.sleep(0.5)
    remote_client.set_registered_model_alias(REGISTERED_MODEL, "Production", registered.version)

    url = f"https://dagshub.com/{DAGSHUB_OWNER}/{DAGSHUB_REPO}/experiments"
    print("Evidencia (experimentos): " + url)
    print(f"Modelo remoto registrado: {REGISTERED_MODEL} v{registered.version} -> alias Production")

    mlflow.set_tracking_uri(TRACKING_URI)
    print("Tracking de nuevo en local:", TRACKING_URI)
else:
    print("Sin DAGSHUB_TOKEN/DAGSHUB_REPO en .env: se omite la parte remota.")
    print("Paso: https://dagshub.com -> cuenta, repo y Settings > User settings > Tokens.")

Accessing as jaimeramos124

Initialized MLflow to track repo "jaimeramos124/ML2_Series_de_tiempo"

Repository jaimeramos124/ML2_Series_de_tiempo initialized!

Registered model 'demand_forecast' already exists. Creating a new version of this model...
2026/08/12 17:47:10 WARNING mlflow.tracking._model_registry.fluent: Run with id 84b566f38a29408ea069ce3dccbc167a has no artifacts at artifact path 'model', registering model based on models:/m-1182ebf9e1bf42e5b0fe06415aadb3f0 instead


Evidencia (experimentos): https://dagshub.com/jaimeramos124/ML2_Series_de_tiempo/experiments
Modelo remoto registrado: demand_forecast v2 -> alias Production
Tracking de nuevo en local: sqlite:///D:/Jaime Ramos 2/00 Entorno Visual code/ML2_Series_de_tiempo/mlruns/mlflow.db


Created version '2' of model 'demand_forecast'.


## Conclusiones

- El modelo de producción se consume como **pyfunc** vía `models:/demand_forecast@Production`,
  sin acordarse de qué familia ganó; la flavor `pyfunc` está declarada en el `MLmodel`.
- Para el horizonte de 90 días se usa **pronóstico recursivo** día a día: las predicciones
  del día previo alimentan los lags/rolling del siguiente, evitando NaN en las features
  (una sola pasada dejaría `lag_1` indefinido desde el día 2).
- Se genera `reports/submissions/submission_production.csv` con las 13,500 predicciones
  (150 series × 90 días, ene-mar 2018). Es una submission **del subconjunto** (el
  `test.csv` original tiene 45,000 filas para las 500 series).
- La parte remota (DagsHub) queda lista: al configurar `DAGSHUB_OWNER`/`DAGSHUB_REPO`/
  `DAGSHUB_TOKEN` en `.env` y re-ejecutar, se suben los runs **y se registra el modelo de
  producción con alias `Production`** al link público de evidencia.